In [1]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score, accuracy_score,
    classification_report, confusion_matrix
)
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

CACHE_DIR   = Path("../embeddings/cache")
SUBSETS_DIR = Path("../data/subsets")
OUTPUT_DIR  = Path("../data/subsets/downstream_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("All imports successful")

All imports successful


In [2]:
# Cell 2 — Load embeddings and manifests
embeddings_full = np.load(CACHE_DIR / "embeddings_resnet18.npy")
index_full      = pd.read_csv(CACHE_DIR / "embedding_index.csv")
audit_df        = pd.read_csv(SUBSETS_DIR / "label_audit_scores.csv")
cleaned_df      = pd.read_csv(SUBSETS_DIR / "cleaned_manifest.csv")
downweight_df   = pd.read_csv(SUBSETS_DIR / "downweight_config.csv")

print("Embeddings shape :", embeddings_full.shape)
print("Full index shape :", index_full.shape)
print("Audit df shape   :", audit_df.shape)
print("Cleaned df shape :", cleaned_df.shape)

Embeddings shape : (17934, 512)
Full index shape : (17934, 7)
Audit df shape   : (17934, 22)
Cleaned df shape : (16511, 22)


In [ ]:
# Cell 3 — Build dataset for each experiment
# Mapping every image to its embedding row

def build_experiment_data(manifest_df, embeddings_full, index_full):
    """
    Given a manifest, return aligned embeddings and labels.
    Filters to only images present in the manifest.
    """
    valid_ids  = set(manifest_df["image_id"].values)
    id_to_label = dict(zip(
        manifest_df["image_id"], manifest_df["label"]
    ))

    rows       = []
    labels_out = []

    for idx, row in index_full.iterrows():
        img_id = row["image_id"]
        if img_id in valid_ids:
            rows.append(idx)
            labels_out.append(id_to_label[img_id])

    X = embeddings_full[rows]
    y = np.array(labels_out)

    return X, y, np.array(rows)


# Experiment 1 — NOISY: all images, original labels
X_noisy, y_noisy, _ = build_experiment_data(
    audit_df, embeddings_full, index_full
)
print(f"Experiment 1 — Noisy   : {X_noisy.shape}")

# Experiment 2 — CLEANED
X_clean, y_clean, _ = build_experiment_data(
    cleaned_df, embeddings_full, index_full
)
print(f"Experiment 2 — Cleaned : {X_clean.shape}")

# Experiment 3 — DOWNWEIGHTED: all images with downweight
# (used in weighted training below)
X_downwt, y_downwt, rows_downwt = build_experiment_data(
    audit_df, embeddings_full, index_full
)

# Replace the sample_weights line with this
weights = downweight_df.drop_duplicates(
    subset="image_id"
).set_index("image_id")["loss_weight"]

sample_weights = np.array([
    float(weights.get(index_full.iloc[r]["image_id"], 1.0))
    for r in rows_downwt
])

print(f"Experiment 3 — Downwt  : {X_downwt.shape}")
print(f"Sample weights shape   : {sample_weights.shape}")
print(f"Weight range: {sample_weights.min():.3f} to {sample_weights.max():.3f}")

Experiment 1 — Noisy   : (17934, 512)
Experiment 2 — Cleaned : (16920, 512)
Experiment 3 — Downwt  : (17934, 512)
Sample weights shape   : (17934,)
Weight range: 0.000 to 1.000


In [ ]:
# Cell 4 — Encode labels consistently
# Useing same encoder across all experiments so F1 scores are comparable

all_labels = np.concatenate([y_noisy, y_clean])
le = LabelEncoder()
le.fit(all_labels)

y_noisy_enc  = le.transform(y_noisy)
y_clean_enc  = le.transform(y_clean)
y_downwt_enc = le.transform(y_downwt)

print("Classes:", le.classes_.tolist())
print("Number of classes:", len(le.classes_))

Classes: ['BCC', 'BKL', 'Consolidation', 'MEL', 'NV', 'No Finding']
Number of classes: 6


In [ ]:
# Cell 5 — Cross-validation evaluation function
# Uses 5-fold stratified CV for reliable F1 estimates
# This Returns mean and std across folds

def evaluate_with_cv(X, y, sample_weight=None,
                     n_splits=5, experiment_name=""):
    skf     = StratifiedKFold(
        n_splits=n_splits, shuffle=True, random_state=42
    )
    f1_scores    = []
    acc_scores   = []
    fold_reports = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Apply sample weights to training fold only
        if sample_weight is not None:
            fold_weights = sample_weight[train_idx]
        else:
            fold_weights = None

        clf = LogisticRegression(
            max_iter=1000,
            random_state=42,
            n_jobs=-1,
            C=1.0
        )
        clf.fit(X_train, y_train,
                sample_weight=fold_weights)

        y_pred = clf.predict(X_test)

        f1  = f1_score(y_test, y_pred,
                       average="macro", zero_division=0)
        acc = accuracy_score(y_test, y_pred)

        f1_scores.append(f1)
        acc_scores.append(acc)

        fold_reports.append({
            "fold"    : fold + 1,
            "f1_macro": round(f1, 4),
            "accuracy": round(acc, 4)
        })

        print(f"  {experiment_name} Fold {fold+1}: "
              f"F1={f1:.4f}  Acc={acc:.4f}")

    mean_f1  = np.mean(f1_scores)
    std_f1   = np.std(f1_scores)
    mean_acc = np.mean(acc_scores)

    return {
        "mean_f1"     : round(mean_f1, 4),
        "std_f1"      : round(std_f1, 4),
        "mean_acc"    : round(mean_acc, 4),
        "fold_reports": fold_reports
    }


print("Evaluation function defined.")
print("Using 5-fold stratified cross-validation.")

Evaluation function defined.
Using 5-fold stratified cross-validation.


In [6]:
# Cell 6 — Experiment 1: Baseline on NOISY data
print("=" * 55)
print("EXPERIMENT 1 — BASELINE ON NOISY DATA")
print("=" * 55)
print(f"Training on {len(y_noisy_enc)} images "
      f"(all original, noisy labels)\n")

results_noisy = evaluate_with_cv(
    X_noisy, y_noisy_enc,
    sample_weight=None,
    experiment_name="Noisy"
)

print(f"\nNoisy Baseline — Mean Macro F1 : "
      f"{results_noisy['mean_f1']:.4f} "
      f"± {results_noisy['std_f1']:.4f}")
print(f"Noisy Baseline — Mean Accuracy : "
      f"{results_noisy['mean_acc']:.4f}")

EXPERIMENT 1 — BASELINE ON NOISY DATA
Training on 17934 images (all original, noisy labels)

  Noisy Fold 1: F1=0.6260  Acc=0.8138
  Noisy Fold 2: F1=0.6278  Acc=0.8068
  Noisy Fold 3: F1=0.6181  Acc=0.7996
  Noisy Fold 4: F1=0.6174  Acc=0.8074
  Noisy Fold 5: F1=0.6198  Acc=0.8104

Noisy Baseline — Mean Macro F1 : 0.6218 ± 0.0043
Noisy Baseline — Mean Accuracy : 0.8076


In [7]:
# Cell 7 — Experiment 2: Baseline on CLEANED data
print("=" * 55)
print("EXPERIMENT 2 — BASELINE ON CLEANED DATA")
print("=" * 55)
print(f"Training on {len(y_clean_enc)} images "
      f"(353 noisy removed by your auditing system)\n")

results_clean = evaluate_with_cv(
    X_clean, y_clean_enc,
    sample_weight=None,
    experiment_name="Cleaned"
)

print(f"\nCleaned Baseline — Mean Macro F1 : "
      f"{results_clean['mean_f1']:.4f} "
      f"± {results_clean['std_f1']:.4f}")
print(f"Cleaned Baseline — Mean Accuracy : "
      f"{results_clean['mean_acc']:.4f}")

EXPERIMENT 2 — BASELINE ON CLEANED DATA
Training on 16920 images (353 noisy removed by your auditing system)

  Cleaned Fold 1: F1=0.5959  Acc=0.8008
  Cleaned Fold 2: F1=0.6066  Acc=0.8011
  Cleaned Fold 3: F1=0.6113  Acc=0.8044
  Cleaned Fold 4: F1=0.5843  Acc=0.7923
  Cleaned Fold 5: F1=0.5964  Acc=0.8023

Cleaned Baseline — Mean Macro F1 : 0.5989 ± 0.0094
Cleaned Baseline — Mean Accuracy : 0.8002


In [8]:
# Cell 8 — Experiment 3: Downweighted training
# All 3807 images kept but suspicious ones have reduced loss weight
print("=" * 55)
print("EXPERIMENT 3 — DOWNWEIGHTED TRAINING")
print("=" * 55)
print(f"Training on {len(y_downwt_enc)} images "
      f"(all kept, suspicious images downweighted)\n")

results_downwt = evaluate_with_cv(
    X_downwt, y_downwt_enc,
    sample_weight=sample_weights,
    experiment_name="Downwt"
)

print(f"\nDownweighted — Mean Macro F1 : "
      f"{results_downwt['mean_f1']:.4f} "
      f"± {results_downwt['std_f1']:.4f}")
print(f"Downweighted — Mean Accuracy : "
      f"{results_downwt['mean_acc']:.4f}")

EXPERIMENT 3 — DOWNWEIGHTED TRAINING
Training on 17934 images (all kept, suspicious images downweighted)

  Downwt Fold 1: F1=0.3347  Acc=0.3159
  Downwt Fold 2: F1=0.3338  Acc=0.3170
  Downwt Fold 3: F1=0.2990  Acc=0.3011
  Downwt Fold 4: F1=0.3439  Acc=0.3164
  Downwt Fold 5: F1=0.3360  Acc=0.3176

Downweighted — Mean Macro F1 : 0.3295 ± 0.0157
Downweighted — Mean Accuracy : 0.3136


In [ ]:
# Cell 9 — DivideMix simplified implementation
# Full DivideMix requires a deep network training loop
# This implements the core DivideMix principle:
# GMM on loss values implies -> clean/noisy probability -> weighted training
# Applied to logistic regression for computational feasibility

from sklearn.mixture import GaussianMixture

print("=" * 55)
print("EXPERIMENT 4 — DIVIDEMIX (SIMPLIFIED)")
print("=" * 55)
print(f"Training on {len(y_noisy_enc)} images")
print("Using GMM on loss values to separate clean from noisy\n")

def dividemix_simplified(X, y, n_splits=5,
                          experiment_name="DivideMix"):
    """
    Simplified DivideMix using logistic regression:
    Step 1: Train initial classifier on all data
    Step 2: Compute per-sample loss
    Step 3: Fit GMM to loss distribution
    Step 4: Get clean probability per sample
    Step 5: Retrain with clean probabilities as weights
    Step 6: Evaluate with cross-validation
    """
    skf       = StratifiedKFold(
        n_splits=n_splits, shuffle=True, random_state=42
    )
    f1_scores  = []
    acc_scores = []

    for fold, (train_idx, test_idx) in enumerate(
        skf.split(X, y)
    ):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Step 1 — Initial classifier on all training data
        clf_init = LogisticRegression(
            max_iter=1000, random_state=42, n_jobs=-1
        )
        clf_init.fit(X_train, y_train)

        # Step 2 — Compute per-sample cross-entropy loss
        proba = clf_init.predict_proba(X_train)
        n_classes = proba.shape[1]
        losses = []
        for i, true_label in enumerate(y_train):
            p = proba[i][true_label]
            p = np.clip(p, 1e-7, 1.0)
            losses.append(-np.log(p))
        losses = np.array(losses).reshape(-1, 1)

        # Step 3 — Fit GMM with 2 components to loss distribution
        # Component 1 = clean (low loss)
        # Component 2 = noisy (high loss)
        gmm = GaussianMixture(
            n_components=2,
            random_state=42,
            max_iter=100
        )
        gmm.fit(losses)

        # Step 4 — Extracting probability of belonging to clean component
        # Clean component is the one with lower mean loss 
        gmm_probs  = gmm.predict_proba(losses)
        clean_comp = np.argmin(gmm.means_.flatten())
        clean_prob = gmm_probs[:, clean_comp]

        # Step 5 — Retrain with clean probabilities as weights
        clf_final = LogisticRegression(
            max_iter=1000, random_state=42, n_jobs=-1
        )
        clf_final.fit(
            X_train, y_train,
            sample_weight=clean_prob
        )

        # Step 6 — Evaluate
        y_pred = clf_final.predict(X_test)
        f1  = f1_score(
            y_test, y_pred,
            average="macro", zero_division=0
        )
        acc = accuracy_score(y_test, y_pred)

        f1_scores.append(f1)
        acc_scores.append(acc)

        print(f"  {experiment_name} Fold {fold+1}: "
              f"F1={f1:.4f}  Acc={acc:.4f}  "
              f"Clean prob mean={clean_prob.mean():.3f}")

    return {
        "mean_f1" : round(np.mean(f1_scores), 4),
        "std_f1"  : round(np.std(f1_scores), 4),
        "mean_acc": round(np.mean(acc_scores), 4)
    }


results_dividemix = dividemix_simplified(
    X_noisy, y_noisy_enc
)

print(f"\nDivideMix — Mean Macro F1 : "
      f"{results_dividemix['mean_f1']:.4f} "
      f"± {results_dividemix['std_f1']:.4f}")
print(f"DivideMix — Mean Accuracy : "
      f"{results_dividemix['mean_acc']:.4f}")

EXPERIMENT 4 — DIVIDEMIX (SIMPLIFIED)
Training on 17934 images
Using GMM on loss values to separate clean from noisy

  DivideMix Fold 1: F1=0.4922  Acc=0.7502  Clean prob mean=0.410
  DivideMix Fold 2: F1=0.5050  Acc=0.7586  Clean prob mean=0.409
  DivideMix Fold 3: F1=0.5011  Acc=0.7600  Clean prob mean=0.420
  DivideMix Fold 4: F1=0.4880  Acc=0.7435  Clean prob mean=0.413
  DivideMix Fold 5: F1=0.5234  Acc=0.7658  Clean prob mean=0.417

DivideMix — Mean Macro F1 : 0.5019 ± 0.0123
DivideMix — Mean Accuracy : 0.7556


In [ ]:
# Cell 10 — Per class F1 breakdown
# Train on full data once, evaluate per class results
# This Gives insight into which classes benefit most from cleaning

print("PER-CLASS F1 BREAKDOWN\n")
print("Training final models on 80% split for per-class analysis")

from sklearn.model_selection import train_test_split

def get_per_class_f1(X, y, le, sample_weight=None,
                     test_size=0.2):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=42,
        stratify=y
    )
    clf = LogisticRegression(
        max_iter=1000, random_state=42, n_jobs=-1
    )
    train_weights = None
    if sample_weight is not None:
        n_train = len(X_train)
        train_weights = sample_weight[:n_train]

    clf.fit(X_train, y_train, sample_weight=train_weights)
    y_pred = clf.predict(X_test)

    report = classification_report(
        y_test, y_pred,
        target_names=le.classes_,
        output_dict=True,
        zero_division=0
    )
    return report


report_noisy   = get_per_class_f1(
    X_noisy, y_noisy_enc, le
)
report_clean   = get_per_class_f1(
    X_clean, y_clean_enc, le
)

print(f"\n{'Class':<40} {'Noisy F1':>10} "
      f"{'Cleaned F1':>12} {'Change':>10}")
print("-" * 75)

for cls in le.classes_:
    f1_n = report_noisy.get(cls, {}).get("f1-score", 0)
    f1_c = report_clean.get(cls, {}).get("f1-score", 0)
    change = f1_c - f1_n
    direction = "↑" if change > 0 else "↓" if change < 0 else "→"
    print(f"{cls:<40} {f1_n:>10.4f} "
          f"{f1_c:>12.4f} "
          f"{direction}{abs(change):>8.4f}")

PER-CLASS F1 BREAKDOWN

Training final models on 80% split for per-class analysis

Class                                      Noisy F1   Cleaned F1     Change
---------------------------------------------------------------------------
BCC                                          0.7390       0.7154 ↓  0.0236
BKL                                          0.5962       0.5267 ↓  0.0695
Consolidation                                0.9864       0.9876 ↑  0.0012
MEL                                          0.5697       0.6031 ↑  0.0334
NV                                           0.6953       0.6310 ↓  0.0643
No Finding                                   0.1455       0.1887 ↑  0.0432


In [11]:
# Cell 11 — Final comparison table
print("\n")
print("=" * 65)
print("FINAL COMPARISON — ALL EXPERIMENTS")
print("=" * 65)
print(f"\n{'Method':<35} {'Macro F1':>10} "
      f"{'Std':>8} {'vs Noisy':>12}")
print("-" * 65)

baseline_f1 = results_noisy["mean_f1"]

experiments = [
    ("Baseline (Noisy Data)",
     results_noisy["mean_f1"],
     results_noisy["std_f1"]),
    ("Baseline (Cleaned — Your System)",
     results_clean["mean_f1"],
     results_clean["std_f1"]),
    ("Downweighted Training",
     results_downwt["mean_f1"],
     results_downwt["std_f1"]),
    ("DivideMix (Simplified)",
     results_dividemix["mean_f1"],
     results_dividemix["std_f1"]),
]

for name, f1, std in experiments:
    diff = f1 - baseline_f1
    direction = "+" if diff >= 0 else ""
    print(f"{name:<35} {f1:>10.4f} "
          f"{std:>8.4f} "
          f"{direction}{diff:>10.4f}")

print("\n")
print("Improvement = Macro F1 vs Noisy Baseline")
print("Positive value = better than noisy baseline")
print("Negative value = worse than noisy baseline")



FINAL COMPARISON — ALL EXPERIMENTS

Method                                Macro F1      Std     vs Noisy
-----------------------------------------------------------------
Baseline (Noisy Data)                   0.6218   0.0043 +    0.0000
Baseline (Cleaned — Your System)        0.5989   0.0094    -0.0229
Downweighted Training                   0.3295   0.0157    -0.2923
DivideMix (Simplified)                  0.5019   0.0123    -0.1199


Improvement = Macro F1 vs Noisy Baseline
Positive value = better than noisy baseline
Negative value = worse than noisy baseline


In [12]:
# Cell 12 — Save all results

# Summary table
summary_df = pd.DataFrame([
    {
        "method"     : "Baseline_Noisy",
        "n_images"   : len(y_noisy_enc),
        "mean_f1"    : results_noisy["mean_f1"],
        "std_f1"     : results_noisy["std_f1"],
        "mean_acc"   : results_noisy["mean_acc"],
        "vs_baseline": 0.0
    },
    {
        "method"     : "Baseline_Cleaned",
        "n_images"   : len(y_clean_enc),
        "mean_f1"    : results_clean["mean_f1"],
        "std_f1"     : results_clean["std_f1"],
        "mean_acc"   : results_clean["mean_acc"],
        "vs_baseline": round(
            results_clean["mean_f1"] -
            results_noisy["mean_f1"], 4
        )
    },
    {
        "method"     : "Downweighted",
        "n_images"   : len(y_downwt_enc),
        "mean_f1"    : results_downwt["mean_f1"],
        "std_f1"     : results_downwt["std_f1"],
        "mean_acc"   : results_downwt["mean_acc"],
        "vs_baseline": round(
            results_downwt["mean_f1"] -
            results_noisy["mean_f1"], 4
        )
    },
    {
        "method"     : "DivideMix_Simplified",
        "n_images"   : len(y_noisy_enc),
        "mean_f1"    : results_dividemix["mean_f1"],
        "std_f1"     : results_dividemix["std_f1"],
        "mean_acc"   : results_dividemix["mean_acc"],
        "vs_baseline": round(
            results_dividemix["mean_f1"] -
            results_noisy["mean_f1"], 4
        )
    }
])

summary_df.to_csv(
    OUTPUT_DIR / "experiment_comparison.csv", index=False
)
print("Saved experiment_comparison.csv")
print(summary_df.to_string(index=False))

Saved experiment_comparison.csv
              method  n_images  mean_f1  std_f1  mean_acc  vs_baseline
      Baseline_Noisy     17934   0.6218  0.0043    0.8076       0.0000
    Baseline_Cleaned     16920   0.5989  0.0094    0.8002      -0.0229
        Downweighted     17934   0.3295  0.0157    0.3136      -0.2923
DivideMix_Simplified     17934   0.5019  0.0123    0.7556      -0.1199


In [ ]:
# Cell 13 — Final summary for dissertation
print("\n")
print("=" * 65)
print("NOTEBOOK 10B (IF) — DISSERTATION SUMMARY")
print("=" * 65)

best_method      = summary_df.loc[
    summary_df["mean_f1"].idxmax(), "method"
]
best_f1          = summary_df["mean_f1"].max()
best_improvement = summary_df["vs_baseline"].max()

# Get actual counts from loaded data
n_chexpert = (index_full["dataset"] == "chexpert").sum()
n_isic     = (index_full["dataset"] == "isic").sum()
n_total    = len(index_full)
n_cleaned  = len(cleaned_df)
n_removed  = n_total - n_cleaned

print(f"""
EXPERIMENTAL SETUP:
  Datasets   : CheXpert ({n_chexpert}) + ISIC ({n_isic})
  Total      : {n_total} images
  Embeddings : ResNet18 frozen (512-dim)
  Evaluation : 5-fold stratified cross-validation
  Metric     : Macro F1 (handles class imbalance)
  Removed    : {n_removed} high-suspicion images

RESULTS (Combined Dataset):
  Noisy Baseline F1   : {results_noisy['mean_f1']:.4f}
  Cleaned Baseline F1 : {results_clean['mean_f1']:.4f}
  Downweighted F1     : {results_downwt['mean_f1']:.4f}
  DivideMix F1        : {results_dividemix['mean_f1']:.4f}

BEST METHOD  : {best_method}
BEST F1      : {best_f1:.4f}
IMPROVEMENT  : +{best_improvement:.4f} over noisy baseline

KEY FINDINGS:
  1. Hard removal (your system) changed F1 by
     {results_clean['mean_f1']-results_noisy['mean_f1']:+.4f}
     vs noisy baseline

  2. Downweighting changed F1 by
     {results_downwt['mean_f1']-results_noisy['mean_f1']:+.4f}
     vs noisy baseline

  3. DivideMix changed F1 by
     {results_dividemix['mean_f1']-results_noisy['mean_f1']:+.4f}
     vs noisy baseline
""")



NOTEBOOK 10B COMPLETE — DISSERTATION SUMMARY

EXPERIMENTAL SETUP:
  Datasets   : CheXpert (8776) + ISIC (9158)
  Total      : 17934 images
  Embeddings : ResNet18 frozen (512-dim)
  Evaluation : 5-fold stratified cross-validation
  Metric     : Macro F1 (handles class imbalance)
  Removed    : 1423 high-suspicion images

RESULTS (Combined Dataset):
  Noisy Baseline F1   : 0.6218
  Cleaned Baseline F1 : 0.5989
  Downweighted F1     : 0.3295
  DivideMix F1        : 0.5019

BEST METHOD  : Baseline_Noisy
BEST F1      : 0.6218
IMPROVEMENT  : +0.0000 over noisy baseline

KEY FINDINGS:
  1. Hard removal (your system) changed F1 by
     -0.0229
     vs noisy baseline

  2. Downweighting changed F1 by
     -0.2923
     vs noisy baseline

  3. DivideMix changed F1 by
     -0.1199
     vs noisy baseline



In [14]:
# Cell 14 — Per-dataset setup with MLP
from sklearn.neural_network import MLPClassifier

print("Setting up per-dataset experiments with MLP classifier")
print("MLP architecture: 512 → 256 → 128 → n_classes\n")

def build_dataset_specific_data(manifest_df, embeddings_full,
                                index_full, dataset_name):
    ds_manifest = manifest_df[
        manifest_df["dataset"] == dataset_name
    ].copy()

    valid_ids   = set(ds_manifest["image_id"].values)
    id_to_label = dict(zip(
        ds_manifest["image_id"], ds_manifest["label"]
    ))

    rows       = []
    labels_out = []

    for idx, row in index_full.iterrows():
        img_id = row["image_id"]
        if img_id in valid_ids:
            rows.append(idx)
            labels_out.append(id_to_label[img_id])

    X = embeddings_full[rows]
    y = np.array(labels_out)
    return X, y, np.array(rows)


# Clean weights lookup — deduplicated
w_lookup = downweight_df.drop_duplicates(
    subset="image_id"
).set_index("image_id")["loss_weight"]

datasets = ["chexpert", "isic"]
data     = {}

for ds in datasets:
    # Noisy
    X_n, y_n, rows_n = build_dataset_specific_data(
        audit_df, embeddings_full, index_full, ds
    )
    # Cleaned
    X_c, y_c, rows_c = build_dataset_specific_data(
        cleaned_df, embeddings_full, index_full, ds
    )
    # Downweighted — same as noisy but with weights
    X_d, y_d, rows_d = build_dataset_specific_data(
        audit_df, embeddings_full, index_full, ds
    )
    sw = np.array([
        float(w_lookup.get(
            index_full.iloc[r]["image_id"], 1.0
        ))
        for r in rows_d
    ])

    # Encode labels per dataset
    le_ds = LabelEncoder()
    le_ds.fit(y_n)

    data[ds] = {
        "X_noisy"       : X_n,
        "y_noisy"       : le_ds.transform(y_n),
        "X_clean"       : X_c,
        "y_clean"       : le_ds.transform(y_c),
        "X_downwt"      : X_d,
        "y_downwt"      : le_ds.transform(y_d),
        "sample_weights": sw,
        "le"            : le_ds,
        "n_classes"     : len(le_ds.classes_),
        "classes"       : le_ds.classes_.tolist()
    }

    print(f"{ds.upper()}:")
    print(f"  Noisy  : {X_n.shape[0]} images, "
          f"{len(le_ds.classes_)} classes")
    print(f"  Cleaned: {X_c.shape[0]} images")
    print(f"  Classes: {le_ds.classes_.tolist()}\n")

Setting up per-dataset experiments with MLP classifier
MLP architecture: 512 → 256 → 128 → n_classes

CHEXPERT:
  Noisy  : 8776 images, 2 classes
  Cleaned: 8776 images
  Classes: ['Consolidation', 'No Finding']

ISIC:
  Noisy  : 9158 images, 4 classes
  Cleaned: 8144 images
  Classes: ['BCC', 'BKL', 'MEL', 'NV']



In [15]:
# Cell 15 — MLP evaluation function
# Option 4: MLP with hidden layers instead of logistic regression

def evaluate_mlp(X, y, sample_weight=None,
                 n_splits=5, experiment_name=""):
    """
    5-fold CV evaluation using MLP classifier
    Architecture: 512 → 256 → 128 → n_classes
    """
    skf        = StratifiedKFold(
        n_splits=n_splits, shuffle=True, random_state=42
    )
    f1_scores  = []
    acc_scores = []

    for fold, (train_idx, test_idx) in enumerate(
        skf.split(X, y)
    ):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        fold_weights = (
            sample_weight[train_idx]
            if sample_weight is not None else None
        )

        clf = MLPClassifier(
            hidden_layer_sizes=(256, 128),
            activation="relu",
            max_iter=300,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=15,
            learning_rate_init=0.001
        )
        clf.fit(X_train, y_train,
                sample_weight=fold_weights)

        y_pred = clf.predict(X_test)
        f1  = f1_score(
            y_test, y_pred,
            average="macro", zero_division=0
        )
        acc = accuracy_score(y_test, y_pred)

        f1_scores.append(f1)
        acc_scores.append(acc)

        print(f"  {experiment_name} Fold {fold+1}: "
              f"F1={f1:.4f}  Acc={acc:.4f}")

    return {
        "mean_f1" : round(np.mean(f1_scores), 4),
        "std_f1"  : round(np.std(f1_scores), 4),
        "mean_acc": round(np.mean(acc_scores), 4)
    }


print("MLP evaluation function defined.")
print("Architecture: 512 → 256 → 128 → n_classes")
print("Using early stopping to prevent overfitting")

MLP evaluation function defined.
Architecture: 512 → 256 → 128 → n_classes
Using early stopping to prevent overfitting


In [16]:
# Cell 16 — DivideMix with MLP per dataset

def dividemix_mlp(X, y, n_splits=5,
                  experiment_name="DivideMix-MLP"):
    """
    DivideMix using MLP:
    Step 1: Train initial MLP on all data
    Step 2: Compute per-sample cross-entropy loss
    Step 3: Fit GMM to separate clean from noisy
    Step 4: Retrain MLP with clean probabilities as weights
    """
    skf        = StratifiedKFold(
        n_splits=n_splits, shuffle=True, random_state=42
    )
    f1_scores  = []
    acc_scores = []

    for fold, (train_idx, test_idx) in enumerate(
        skf.split(X, y)
    ):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Step 1 — Initial MLP
        clf_init = MLPClassifier(
            hidden_layer_sizes=(256, 128),
            activation="relu",
            max_iter=200,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10
        )
        clf_init.fit(X_train, y_train)

        # Step 2 — Per-sample loss
        proba  = clf_init.predict_proba(X_train)
        losses = []
        for i, true_label in enumerate(y_train):
            p = np.clip(proba[i][true_label], 1e-7, 1.0)
            losses.append(-np.log(p))
        losses = np.array(losses).reshape(-1, 1)

        # Step 3 — GMM separation
        gmm = GaussianMixture(
            n_components=2,
            random_state=42,
            max_iter=100
        )
        gmm.fit(losses)
        gmm_probs  = gmm.predict_proba(losses)
        clean_comp = np.argmin(gmm.means_.flatten())
        clean_prob = gmm_probs[:, clean_comp]

        # Step 4 — Retrain MLP with clean weights
        clf_final = MLPClassifier(
            hidden_layer_sizes=(256, 128),
            activation="relu",
            max_iter=300,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=15
        )
        clf_final.fit(
            X_train, y_train,
            sample_weight=clean_prob
        )

        y_pred = clf_final.predict(X_test)
        f1  = f1_score(
            y_test, y_pred,
            average="macro", zero_division=0
        )
        acc = accuracy_score(y_test, y_pred)

        f1_scores.append(f1)
        acc_scores.append(acc)

        print(f"  {experiment_name} Fold {fold+1}: "
              f"F1={f1:.4f}  Acc={acc:.4f}  "
              f"Clean prob={clean_prob.mean():.3f}")

    return {
        "mean_f1" : round(np.mean(f1_scores), 4),
        "std_f1"  : round(np.std(f1_scores), 4),
        "mean_acc": round(np.mean(acc_scores), 4)
    }


print("DivideMix MLP function defined.")

DivideMix MLP function defined.


In [17]:
# Cell 17 — Run all experiments for CHEXPERT

print("=" * 60)
print("CHEXPERT — ALL EXPERIMENTS (MLP)")
print("=" * 60)

ds     = "chexpert"
ds_data = data[ds]

print(f"\nDataset: {ds.upper()}")
print(f"Classes: {ds_data['classes']}\n")

# Experiment 1 — Noisy baseline
print("--- Experiment 1: Noisy Baseline ---")
chex_noisy = evaluate_mlp(
    ds_data["X_noisy"],
    ds_data["y_noisy"],
    experiment_name="CheX-Noisy"
)
print(f"Result: F1={chex_noisy['mean_f1']:.4f} "
      f"± {chex_noisy['std_f1']:.4f}\n")

# Experiment 2 — Cleaned
print("--- Experiment 2: Cleaned Baseline ---")
chex_clean = evaluate_mlp(
    ds_data["X_clean"],
    ds_data["y_clean"],
    experiment_name="CheX-Clean"
)
print(f"Result: F1={chex_clean['mean_f1']:.4f} "
      f"± {chex_clean['std_f1']:.4f}\n")

# Experiment 3 — Downweighted
print("--- Experiment 3: Downweighted ---")

# Fix: clip weights so none are exactly 0
# Removed images get minimum weight 0.1 instead of 0.0
clipped_weights = np.clip(ds_data["sample_weights"], 0.1, 1.0)

chex_downwt = evaluate_mlp(
    ds_data["X_downwt"],
    ds_data["y_downwt"],
    sample_weight=clipped_weights,
    experiment_name="CheX-Downwt"
)
print(f"Result: F1={chex_downwt['mean_f1']:.4f} "
      f"± {chex_downwt['std_f1']:.4f}\n")

# Experiment 4 — DivideMix
print("--- Experiment 4: DivideMix ---")
chex_dividemix = dividemix_mlp(
    ds_data["X_noisy"],
    ds_data["y_noisy"],
    experiment_name="CheX-DivideMix"
)
print(f"Result: F1={chex_dividemix['mean_f1']:.4f} "
      f"± {chex_dividemix['std_f1']:.4f}")

CHEXPERT — ALL EXPERIMENTS (MLP)

Dataset: CHEXPERT
Classes: ['Consolidation', 'No Finding']

--- Experiment 1: Noisy Baseline ---
  CheX-Noisy Fold 1: F1=0.5795  Acc=0.9778
  CheX-Noisy Fold 2: F1=0.4950  Acc=0.9801
  CheX-Noisy Fold 3: F1=0.4948  Acc=0.9795
  CheX-Noisy Fold 4: F1=0.4948  Acc=0.9795
  CheX-Noisy Fold 5: F1=0.5579  Acc=0.9766
Result: F1=0.5244 ± 0.0368

--- Experiment 2: Cleaned Baseline ---
  CheX-Clean Fold 1: F1=0.5795  Acc=0.9778
  CheX-Clean Fold 2: F1=0.4950  Acc=0.9801
  CheX-Clean Fold 3: F1=0.4948  Acc=0.9795
  CheX-Clean Fold 4: F1=0.4948  Acc=0.9795
  CheX-Clean Fold 5: F1=0.5579  Acc=0.9766
Result: F1=0.5244 ± 0.0368

--- Experiment 3: Downweighted ---
  CheX-Downwt Fold 1: F1=0.5690  Acc=0.9744
  CheX-Downwt Fold 2: F1=0.4950  Acc=0.9801
  CheX-Downwt Fold 3: F1=0.4948  Acc=0.9795
  CheX-Downwt Fold 4: F1=0.5954  Acc=0.9818
  CheX-Downwt Fold 5: F1=0.6035  Acc=0.9795
Result: F1=0.5515 ± 0.0476

--- Experiment 4: DivideMix ---
  CheX-DivideMix Fold 1: F1=0

In [18]:
# Cell 18 — Run all experiments for ISIC

print("=" * 60)
print("ISIC — ALL EXPERIMENTS (MLP)")
print("=" * 60)

ds      = "isic"
ds_data = data[ds]

print(f"\nDataset: {ds.upper()}")
print(f"Classes: {ds_data['classes']}\n")

# Experiment 1 — Noisy baseline
print("--- Experiment 1: Noisy Baseline ---")
isic_noisy = evaluate_mlp(
    ds_data["X_noisy"],
    ds_data["y_noisy"],
    experiment_name="ISIC-Noisy"
)
print(f"Result: F1={isic_noisy['mean_f1']:.4f} "
      f"± {isic_noisy['std_f1']:.4f}\n")

# Experiment 2 — Cleaned
print("--- Experiment 2: Cleaned Baseline ---")
isic_clean = evaluate_mlp(
    ds_data["X_clean"],
    ds_data["y_clean"],
    experiment_name="ISIC-Clean"
)
print(f"Result: F1={isic_clean['mean_f1']:.4f} "
      f"± {isic_clean['std_f1']:.4f}\n")

# Experiment 3 — Downweighted
print("--- Experiment 3: Downweighted ---")
clipped_weights = np.clip(ds_data["sample_weights"], 0.1, 1.0)

isic_downwt = evaluate_mlp(
    ds_data["X_downwt"],
    ds_data["y_downwt"],
    sample_weight=clipped_weights,
    experiment_name="ISIC-Downwt"
)
print(f"Result: F1={isic_downwt['mean_f1']:.4f} "
      f"± {isic_downwt['std_f1']:.4f}\n")

# Experiment 4 — DivideMix
print("--- Experiment 4: DivideMix ---")
isic_dividemix = dividemix_mlp(
    ds_data["X_noisy"],
    ds_data["y_noisy"],
    experiment_name="ISIC-DivideMix"
)
print(f"Result: F1={isic_dividemix['mean_f1']:.4f} "
      f"± {isic_dividemix['std_f1']:.4f}")

ISIC — ALL EXPERIMENTS (MLP)

Dataset: ISIC
Classes: ['BCC', 'BKL', 'MEL', 'NV']

--- Experiment 1: Noisy Baseline ---
  ISIC-Noisy Fold 1: F1=0.6778  Acc=0.6850
  ISIC-Noisy Fold 2: F1=0.6844  Acc=0.6921
  ISIC-Noisy Fold 3: F1=0.7089  Acc=0.7162
  ISIC-Noisy Fold 4: F1=0.7250  Acc=0.7286
  ISIC-Noisy Fold 5: F1=0.6764  Acc=0.6849
Result: F1=0.6945 ± 0.0192

--- Experiment 2: Cleaned Baseline ---
  ISIC-Clean Fold 1: F1=0.6621  Acc=0.6685
  ISIC-Clean Fold 2: F1=0.6654  Acc=0.6697
  ISIC-Clean Fold 3: F1=0.6695  Acc=0.6728
  ISIC-Clean Fold 4: F1=0.6668  Acc=0.6685
  ISIC-Clean Fold 5: F1=0.6827  Acc=0.6861
Result: F1=0.6693 ± 0.0071

--- Experiment 3: Downweighted ---
  ISIC-Downwt Fold 1: F1=0.6625  Acc=0.6670
  ISIC-Downwt Fold 2: F1=0.7051  Acc=0.7085
  ISIC-Downwt Fold 3: F1=0.6847  Acc=0.6894
  ISIC-Downwt Fold 4: F1=0.6861  Acc=0.6920
  ISIC-Downwt Fold 5: F1=0.6836  Acc=0.6876
Result: F1=0.6844 ± 0.0135

--- Experiment 4: DivideMix ---
  ISIC-DivideMix Fold 1: F1=0.6596  Acc=0

In [19]:
# Cell 19 — Per-dataset comparison tables

for ds_name, results in [
    ("CHEXPERT", {
        "Noisy Baseline"   : chex_noisy,
        "Cleaned (Yours)"  : chex_clean,
        "Downweighted"     : chex_downwt,
        "DivideMix"        : chex_dividemix
    }),
    ("ISIC", {
        "Noisy Baseline"   : isic_noisy,
        "Cleaned (Yours)"  : isic_clean,
        "Downweighted"     : isic_downwt,
        "DivideMix"        : isic_dividemix
    })
]:
    baseline = results["Noisy Baseline"]["mean_f1"]
    print(f"\n{'='*60}")
    print(f"{ds_name} RESULTS SUMMARY")
    print(f"{'='*60}")
    print(f"{'Method':<25} {'F1':>8} {'Std':>8} {'vs Noisy':>10}")
    print(f"{'-'*55}")
    for method, res in results.items():
        f1   = res["mean_f1"]
        std  = res["std_f1"]
        diff = f1 - baseline
        sign = "+" if diff >= 0 else ""
        print(f"{method:<25} {f1:>8.4f} {std:>8.4f} "
              f"{sign}{diff:>8.4f}")


CHEXPERT RESULTS SUMMARY
Method                          F1      Std   vs Noisy
-------------------------------------------------------
Noisy Baseline              0.5244   0.0368 +  0.0000
Cleaned (Yours)             0.5244   0.0368 +  0.0000
Downweighted                0.5515   0.0476 +  0.0271
DivideMix                   0.4948   0.0001  -0.0296

ISIC RESULTS SUMMARY
Method                          F1      Std   vs Noisy
-------------------------------------------------------
Noisy Baseline              0.6945   0.0192 +  0.0000
Cleaned (Yours)             0.6693   0.0071  -0.0252
Downweighted                0.6844   0.0135  -0.0101
DivideMix                   0.6732   0.0190  -0.0213


In [20]:
# Cell 20 — Per-class F1 analysis per dataset

print("PER-CLASS F1 — NOISY vs CLEANED (MLP)\n")

from sklearn.model_selection import train_test_split

for ds_name, ds_key, X_n, y_n, X_c, y_c, le_ds in [
    ("CHEXPERT", "chexpert",
     data["chexpert"]["X_noisy"],
     data["chexpert"]["y_noisy"],
     data["chexpert"]["X_clean"],
     data["chexpert"]["y_clean"],
     data["chexpert"]["le"]),
    ("ISIC", "isic",
     data["isic"]["X_noisy"],
     data["isic"]["y_noisy"],
     data["isic"]["X_clean"],
     data["isic"]["y_clean"],
     data["isic"]["le"])
]:
    print(f"\n{ds_name}:")
    print(f"{'Class':<40} {'Noisy F1':>10} "
          f"{'Clean F1':>10} {'Change':>10}")
    print("-" * 72)

    # Train on 80%, test on 20%
    Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(
        X_n, y_n, test_size=0.2,
        random_state=42, stratify=y_n
    )
    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
        X_c, y_c, test_size=0.2,
        random_state=42, stratify=y_c
    )

    # MLP noisy
    clf_n = MLPClassifier(
        hidden_layer_sizes=(256, 128),
        max_iter=300, random_state=42,
        early_stopping=True
    )
    clf_n.fit(Xn_tr, yn_tr)

    # MLP clean
    clf_c = MLPClassifier(
        hidden_layer_sizes=(256, 128),
        max_iter=300, random_state=42,
        early_stopping=True
    )
    clf_c.fit(Xc_tr, yc_tr)

    rep_n = classification_report(
        yn_te, clf_n.predict(Xn_te),
        target_names=le_ds.classes_,
        output_dict=True, zero_division=0
    )
    rep_c = classification_report(
        yc_te, clf_c.predict(Xc_te),
        target_names=le_ds.classes_,
        output_dict=True, zero_division=0
    )

    for cls in le_ds.classes_:
        f1_n = rep_n.get(cls, {}).get("f1-score", 0)
        f1_c = rep_c.get(cls, {}).get("f1-score", 0)
        diff = f1_c - f1_n
        arrow = "↑" if diff > 0.01 else "↓" if diff < -0.01 else "→"
        print(f"{cls:<40} {f1_n:>10.4f} "
              f"{f1_c:>10.4f} "
              f"{arrow}{abs(diff):>8.4f}")

PER-CLASS F1 — NOISY vs CLEANED (MLP)


CHEXPERT:
Class                                      Noisy F1   Clean F1     Change
------------------------------------------------------------------------
Consolidation                                0.9896     0.9896 →  0.0000
No Finding                                   0.0526     0.0526 →  0.0000

ISIC:
Class                                      Noisy F1   Clean F1     Change
------------------------------------------------------------------------
BCC                                          0.7721     0.7500 ↓  0.0221
BKL                                          0.6441     0.6344 →  0.0098
MEL                                          0.6543     0.6609 →  0.0066
NV                                           0.7171     0.6400 ↓  0.0771


In [21]:
# Cell 21 — Save all per-dataset results

per_dataset_results = []

for ds_name, noisy, clean, downwt, dividemix in [
    ("chexpert",
     chex_noisy, chex_clean,
     chex_downwt, chex_dividemix),
    ("isic",
     isic_noisy, isic_clean,
     isic_downwt, isic_dividemix)
]:
    baseline = noisy["mean_f1"]
    for method, res in [
        ("Noisy_Baseline", noisy),
        ("Cleaned_YourSystem", clean),
        ("Downweighted", downwt),
        ("DivideMix_MLP", dividemix)
    ]:
        per_dataset_results.append({
            "dataset"    : ds_name,
            "method"     : method,
            "mean_f1"    : res["mean_f1"],
            "std_f1"     : res["std_f1"],
            "mean_acc"   : res["mean_acc"],
            "vs_baseline": round(res["mean_f1"] - baseline, 4)
        })

results_df = pd.DataFrame(per_dataset_results)
results_df.to_csv(
    OUTPUT_DIR / "per_dataset_comparison.csv", index=False
)
print("Saved per_dataset_comparison.csv")
print(results_df.to_string(index=False))

Saved per_dataset_comparison.csv
 dataset             method  mean_f1  std_f1  mean_acc  vs_baseline
chexpert     Noisy_Baseline   0.5244  0.0368    0.9787       0.0000
chexpert Cleaned_YourSystem   0.5244  0.0368    0.9787       0.0000
chexpert       Downweighted   0.5515  0.0476    0.9790       0.0271
chexpert      DivideMix_MLP   0.4948  0.0001    0.9796      -0.0296
    isic     Noisy_Baseline   0.6945  0.0192    0.7014       0.0000
    isic Cleaned_YourSystem   0.6693  0.0071    0.6731      -0.0252
    isic       Downweighted   0.6844  0.0135    0.6889      -0.0101
    isic      DivideMix_MLP   0.6732  0.0190    0.6809      -0.0213


In [22]:
# Cell 22 — Final dissertation summary

print("=" * 65)
print("FINAL DISSERTATION SUMMARY — PER DATASET MLP RESULTS")
print("=" * 65)

for ds in ["chexpert", "isic"]:
    ds_results = results_df[results_df["dataset"] == ds]
    baseline   = ds_results[
        ds_results["method"] == "Noisy_Baseline"
    ]["mean_f1"].values[0]
    best_row   = ds_results.loc[
        ds_results["mean_f1"].idxmax()
    ]

    print(f"\n{ds.upper()}:")
    print(f"  Noisy Baseline F1 : {baseline:.4f}")
    for _, row in ds_results.iterrows():
        diff  = row["mean_f1"] - baseline
        sign  = "+" if diff >= 0 else ""
        print(f"  {row['method']:<25}: "
              f"F1={row['mean_f1']:.4f} "
              f"({sign}{diff:.4f})")
    print(f"  Best method: {best_row['method']} "
          f"(F1={best_row['mean_f1']:.4f})")

FINAL DISSERTATION SUMMARY — PER DATASET MLP RESULTS

CHEXPERT:
  Noisy Baseline F1 : 0.5244
  Noisy_Baseline           : F1=0.5244 (+0.0000)
  Cleaned_YourSystem       : F1=0.5244 (+0.0000)
  Downweighted             : F1=0.5515 (+0.0271)
  DivideMix_MLP            : F1=0.4948 (-0.0296)
  Best method: Downweighted (F1=0.5515)

ISIC:
  Noisy Baseline F1 : 0.6945
  Noisy_Baseline           : F1=0.6945 (+0.0000)
  Cleaned_YourSystem       : F1=0.6693 (-0.0252)
  Downweighted             : F1=0.6844 (-0.0101)
  DivideMix_MLP            : F1=0.6732 (-0.0213)
  Best method: Noisy_Baseline (F1=0.6945)
